In [1]:
import os

from gradient_relevance_score import DistilBertAttributor

In [2]:
attributor = DistilBertAttributor(model_path="./results/distilbert/checkpoint-170")

In [3]:
text = "Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet been secured. At the British Championship in Liverpool, he almost repeated his performance of the previous year, by taking a share of third place. He was the British under-21 Champion each consecutive year between 2005 and 2008. He became a grandmaster on 1 August 2009. He has been one of the co-presenters of the chess podcast The Full English Breakfast since its inaugural show in October 2010."

In [ ]:
target, important_tokens = attributor.compute_attributions(text, merge_scores=False)

In [8]:
text_with_attributions = text.replace("[","").replace("]","").lower()
for token, _ in important_tokens[:10]:
    text_with_attributions = text_with_attributions.replace(" {} ".format(token), " [{}] ".format(token))
text_with_attributions

'stephen j. gordon (born 4 september 1986) [is] a [chess] [grandmaster] from oldham, greater manchester, england. in september [2004] he took a break from his a-level studies at the blue coat school, oldham to compete in the thirteenth monarch assurance isle of man international. in 2005, while still a [fide] master, he finished 6th in the british championships ahead of a [grandmaster] and several international masters. at the [eu] individual open [chess] championship held at liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner nigel short and level with luke mcshane [among] others. probably his best result to date however, was second place in the 2007 british championship, narrowly losing his share of the lead in the final round. in previous rounds, he defeated both tournament victor jacob aagaard and previous champion jonathan rowson. by 2008, his rating had reached [grandmaster] level, although the titl

In [12]:
import requests, json

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
API_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-3.5-turbo"  # Change to "gpt-4" if needed

HEADERS = {
    "Authorization": f"Bearer {OPENAI_API_KEY}",
    "Content-Type": "application/json"
}

In [16]:
def _send_request(messages, api=API_URL, headers=HEADERS, model=MODEL):
    """Internal helper to send request to OpenAI Chat API."""
    payload = {
        "model": model,
        "messages": messages
    }

    response = requests.post(api, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    reply = data["choices"][0]["message"]["content"]
    return reply

In [18]:
prompt = "Explain why the following text is {}anonymized. Words in square brackets [] are important words for the classification of this text. This is the text: ".format("not " if target == 0 else "")
messages = [{"role": "user", "content": prompt + text_with_attributions}]
_send_request(messages)

'The text is not anonymized because it provides specific personal information about Stephen J. Gordon, such as his full name, date of birth, place of birth, and achievements in the field of chess. These details are not generalized or disguised in any way, making it easy to identify the individual being discussed. Additionally, the text mentions specific events, tournaments, and results that are directly associated with Stephen J. Gordon, further compromising the anonymity of the text.'